In [1]:
#Load Libraries
library(Seurat)
library(Signac)
library(GenomeInfoDb)
library(EnsDb.Hsapiens.v86)
library(BSgenome.Hsapiens.UCSC.hg38)
library(AnnotationHub)
library(GenomicRanges)
library(BiocParallel)
library(chromVAR)
library(JASPAR2024)
library(TFBSTools)
library(cicero)
library(ggplot2)
library(patchwork)
library(reticulate)
library(sceasy)
library(future)
library(Matrix)

#Set Options
options(future.globals.maxSize = 400000 * 1024^2) #for 300GB max size
plan("multicore", workers = 4)

register(SerialParam()) 
set.seed(1234)

#Set working directory
setwd("/storage1/fs1/jmillman/Active/DigitalTwin")

Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t


Loading required package: BiocGenerics

Loading required package: generics


Attaching package: ‘generics’


The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, saveRDS, table, tapply, unique,
    unsplit, which.max, which.min


Loadi

In [ ]:
# Load Data
obj<-readRDS("checkpoints/MultiomeAtlas_merged.rds")
obj

# Preprocess and Integrate RNA Data

In [ ]:
DefaultAssay(obj) <- "RNA"
obj <- JoinLayers(obj)
obj[["RNA"]] <- split(obj[["RNA"]], f = obj$orig.ident)

obj <- NormalizeData(obj)
obj <- FindVariableFeatures(obj)
obj <- ScaleData(obj)
obj <- RunPCA(obj)

obj <- FindNeighbors(obj, dims = 1:30, reduction = "pca")
obj <- FindClusters(obj, resolution = 0.5, cluster.name = "merged_rna_clusters")

obj <- RunUMAP(obj, dims = 1:30, reduction = "pca", reduction.name = "umap.rna.merged")

In [ ]:
DefaultAssay(obj) <- "RNA"

obj <- IntegrateLayers(
  object = obj, method = CCAIntegration,
  orig.reduction = "pca", new.reduction = "integrated.cca",
  verbose = FALSE
)

obj <- FindNeighbors(obj, reduction = "integrated.cca", dims = 1:30)
obj <- FindClusters(obj, resolution = 0.5, cluster.name = "cca_clusters")
obj <- RunUMAP(obj, reduction = "integrated.cca", dims = 1:30, reduction.name = "umap.integrated.cca")

obj <- IntegrateLayers(
  object = obj, method = RPCAIntegration,
  orig.reduction = "pca", new.reduction = "integrated.rpca",
  verbose = FALSE
)

obj <- FindNeighbors(obj, reduction = "integrated.rpca", dims = 1:30)
obj <- FindClusters(obj, resolution = 0.5, cluster.name = "rpca_clusters")
obj <- RunUMAP(obj, reduction = "integrated.rpca", dims = 1:30, reduction.name = "umap.integrated.rpca")

obj <- IntegrateLayers(
  object = obj, method = HarmonyIntegration,
  orig.reduction = "pca", new.reduction = "harmony",
  verbose = FALSE,
)

obj <- FindNeighbors(obj, reduction = "harmony", dims = 1:30)
obj <- FindClusters(obj, resolution = 0.5, cluster.name = "harmony_clusters")
obj <- RunUMAP(obj, reduction = "harmony", dims = 1:30, reduction.name = "umap.harmony")

saveRDS(obj, file="checkpoints/MultiomeAtlas_integratedRNA.rds")

# Preprocess and Integrate ATAC Data

In [ ]:
DefaultAssay(obj) <- "peaks"

obj <- FindTopFeatures(obj, min.cutoff = 5)
obj <- RunTFIDF(obj)
obj <- RunSVD(obj)

obj <- FindNeighbors(object = obj, reduction = 'lsi', dims = 2:30)
obj <- FindClusters(object = obj, verbose = FALSE, resolution = 0.5, cluster.name = "merged_atac_clusters")

obj <- RunUMAP(object = obj, reduction = 'lsi', dims = 2:30, reduction.name = 'umap.atac.merged')

In [ ]:
DefaultAssay(obj) <- "peaks"
peaks.list <- SplitObject(obj, split.by = "orig.ident")

In [ ]:
# find integration anchors
integration.anchors <- FindIntegrationAnchors(
  object.list = peaks.list,
  anchor.features = rownames(obj),
  reduction = "rlsi",
  dims = 2:40
)

# integrate LSI embeddings
integrated <- IntegrateEmbeddings(
  anchorset = integration.anchors,
  reductions = obj[["lsi"]],
  new.reduction.name = "integrated_lsi",
  dims.to.integrate = 1:40
)

saveRDS(integrated, file="checkpoints/MultiomeAtlas_integratedATAC.rds")

# Merge Integrated RNA and ATAC Data

In [ ]:
ah <- AnnotationHub()

# Search for the Ensembl 98 EnsDb for Homo sapiens on AnnotationHub
query(ah, "EnsDb.Hsapiens.v98")
ensdb_v98 <- ah[["AH75011"]]

# extract gene annotations from EnsDb
annotation <- GetGRangesFromEnsDb(ensdb = ensdb_v98)

# change to UCSC style since the data was mapped to hg38
seqlevels(annotation) <- paste0('chr', seqlevels(annotation))
genome(annotation) <- "hg38"

rm(ah,ensdb_v98)

In [ ]:
obj <- readRDS("checkpoints/MultiomeAtlas_integratedRNA.rds")
integrated <- readRDS("checkpoints/MultiomeAtlas_integratedATAC.rds")

In [ ]:
obj[["integrated.lsi"]] <- integrated[["integrated_lsi"]]

DefaultAssay(obj) <- "RNA"
obj <- JoinLayers(obj)
obj

rm(integrated)

In [ ]:
DefaultAssay(obj) <- "ATAC"

# Peak Calling using MACS2
peaks <- CallPeaks(obj, assay="ATAC", macs2.path = "/storage1/fs1/jmillman/Active/Matt/R_libraries/Conda/envs/PeakCalling_analysis/bin/macs2")

# remove peaks on nonstandard chromosomes and in genomic blacklist regions
peaks <- keepStandardChromosomes(peaks, pruning.mode = "coarse")
peaks <- subsetByOverlaps(x = peaks, ranges = blacklist_hg38_unified, invert = TRUE)

# quantify counts in each peak
macs2counts <- FeatureMatrix(fragments = Fragments(obj), features = peaks, cells = colnames(obj))

In [ ]:
# create a new assay using the MACS2 peak set and add it to the Seurat object
obj[["integrated.peaks"]] <- CreateChromatinAssay(counts = macs2counts, fragments = Fragments(obj), annotation = annotation)

DefaultAssay(obj) <- "integrated.peaks"
obj <- FindTopFeatures(obj, min.cutoff = 5)
obj <- RunTFIDF(obj)
obj <- RunSVD(obj)

In [ ]:
#Add the gene activity (promoter accessibility) matrix to the Seurat object as a new assay and normalize it
gene.activities <- GeneActivity(obj) 
obj[['GeneActivity']] <- CreateAssayObject(counts = gene.activities)
obj <- NormalizeData(object = obj, assay = 'GeneActivity', normalization.method = 'LogNormalize', scale.factor = median(obj$nCount_GeneActivity))

rm(peaks)
rm(macs2counts)
rm(gene.activities)

# Clustering and Marker Analysis

In [ ]:
Idents(obj) <- "dataset"
obj[["day"]] <- Idents(obj)
Idents(obj) <- "day"

obj <- RenameIdents(object = obj, `s2d1` = "Day05",`s3d1` = "Day07",`s4d2` = "Day10",`s5d1` = "Day13",`s5d3` = "Day15",`s5d5` = "Day17",`s6d1` = "Day20",
                     `s6d7` = "Day26",`s6d14` = "Day33")

obj[["day"]] <- Idents(obj)

In [ ]:
DefaultAssay(obj) <- "RNA"

# build a joint neighbor graph using both assays
obj <- FindMultiModalNeighbors(object = obj,reduction.list = list("harmony", "integrated.lsi"), dims.list = list(1:50, 2:40),modality.weight.name = "RNA.weight", verbose = FALSE)

# build a joint UMAP visualization
obj <- RunUMAP(object = obj, nn.name = "weighted.nn",assay = "RNA",verbose = FALSE, reduction.name = 'umap.integrated.joint')
obj <- FindClusters(obj, graph.name = "wsnn", verbose = FALSE, resolution = 1.6, cluster.name = "intjoint_clusters")

cluster.markers <- FindAllMarkers(object = obj, group.by = 'intjoint_clusters', only.pos = TRUE, min.pct=0.1)
write.csv(cluster.markers, file = "outputs/ClusterMarkers/MultiomeIntegrated_wnnclustermarkers.csv")
rm(cluster.markers)

In [ ]:
#Check differential gene expression over time
DefaultAssay(obj) <- 'RNA'
Idents(obj) <- "day"

differential.expression <- FindAllMarkers(object = obj, only.pos = TRUE)

write.csv(differential.expression,"outputs/ClusterMarkers/MultiomeIntegrated_DEG_day.csv", row.names = TRUE)
rm(differential.expression)
gc()

In [ ]:
DefaultAssay(obj) <- "integrated.peaks"

da_peaks <- FindAllMarkers(object = obj, group.by = "intjoint_clusters", test.use = 'wilcox', min.pct = 0.1)
closest_genes <- ClosestFeature(obj, regions = rownames(da_peaks))
da_peaks <- cbind(da_peaks, closest_genes)

write.csv(da_peaks, file = "outputs/ClusterMarkers/MultiomeIntegrated_wnnclusterpeaks.csv")

In [ ]:
DefaultAssay(obj) <- "integrated.peaks"

da_peaks <- FindAllMarkers(object = obj, group.by = "dataset", test.use = 'wilcox', min.pct = 0.1)
closest_genes <- ClosestFeature(obj, regions = rownames(da_peaks))
da_peaks <- cbind(da_peaks, closest_genes)

write.csv(da_peaks, file = "outputs/ClusterMarkers/MultiomeIntegrated_DAP_day.csv")

# Motif Analysis

In [ ]:
DefaultAssay(obj) <- "integrated.peaks"

# extract position frequency matrices for the motifs
sq24 <- RSQLite::dbConnect(RSQLite::SQLite(), db(JASPAR2024()))
pwm <- getMatrixSet(sq24, list(species = "Homo sapiens", collection = "CORE"))

# add motif information
obj <- AddMotifs(obj, genome = BSgenome.Hsapiens.UCSC.hg38, pfm = pwm)

# add genome info
genome(obj) <- "hg38"

rm(pwm)

In [ ]:
# Computing motif activities
DefaultAssay(obj) <- 'integrated.peaks'
obj <- RunChromVAR(object = obj, genome = BSgenome.Hsapiens.UCSC.hg38)

In [ ]:
#Check differential motif activity by cluster
DefaultAssay(obj) <- 'chromvar'
Idents(obj) <- "intjoint_clusters"

differential.activity <- FindAllMarkers(object = obj, only.pos = TRUE, test.use = 'LR', latent.vars = 'nCount_peaks',slot = "data")
convertIDlist <- ConvertMotifID(object = obj,id = rownames(differential.activity),assay='integrated.peaks')
differential.activity <- cbind(differential.activity,convertIDlist)

write.csv(differential.activity,"outputs/ClusterMarkers/MultiomeIntegrated_wnnclustermotifs.csv", row.names = TRUE)
rm(differential.activity)
gc()

In [ ]:
#Check differential motif activity over time
DefaultAssay(obj) <- 'chromvar'
Idents(obj) <- "day"

differential.activity <- FindAllMarkers(object = obj, only.pos = TRUE, test.use = 'LR', latent.vars = 'nCount_peaks',slot = "data")
convertIDlist <- ConvertMotifID(object = obj,id = rownames(differential.activity),assay='integrated.peaks')
differential.activity <- cbind(differential.activity,convertIDlist)

write.csv(differential.activity,"outputs/ClusterMarkers/MultiomeIntegrated_DAM_day.csv", row.names = TRUE)
rm(differential.activity)
gc()

# Save and Export

In [ ]:
####CHECKPOINT####
saveRDS(obj, file="checkpoints/MultiomeAtlas_integratedFULL.rds")

In [ ]:
peaks <- granges(obj[['integrated.peaks']])
rtracklayer::export.bed(peaks, "rawdata/Multiome/integrated_combined_peaks.bed")

In [ ]:
py_require("anndata")
anndata <- import("anndata")

data.rna <- obj

# convert to v3 assay
DefaultAssay(data.rna) <- 'RNA'
data.rna[["RNA"]] <- as(object = data.rna[["RNA"]], Class = "Assay")

# convert RNA data to  anndata
convertFormat(data.rna, from="seurat", to="anndata", main_layer="counts", assay="RNA", drop_single_values=FALSE, outFile='checkpoints/multiome_rna.h5ad')
rm(data.rna)

In [ ]:
# convert ATAC data to anndata
convertFormat(obj, from="seurat", to="anndata", main_layer="counts", assay="integrated.peaks", drop_single_values=FALSE, 
              outFile='checkpoints/multiome_atac.h5ad')

In [ ]:
sink("Notebooks/1_MultiomeAtlas/02_Multiome_Integration-sessionInfo.txt")
sessionInfo()
sink()